# Phishing URL Dataset - Exploratory Data Analysis

### Team Members
- Muhammad Sarab Aftab
- Saurya Jonchhe
- Nipun Yonjan

For this project, we are using URL characteristics to classify websites as either benign or phishing.

The goal of this notebook is to understand the dataset, check for obvious data problems, and make a basic numeric version that can later be used by our machine learning models.


## Loading the dataset

First I am loading the dataset and checking its overall size and a few sample rows.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("Dataset.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


The dataset should have 116,600 URLs and 26 columns. Most of the columns are already numerical features extracted from each URL, which should make the dataset easier to use for machine learning.


## Looking at the columns

Next I want to see what features are available and what type of data each column contains.


In [ ]:
df.info()


In [ ]:
df.columns.tolist()


Some of the useful features describe the structure of the URL, such as URL length, domain length, number of digits, special characters, subdomains, HTTPS usage, entropy, path length, and query length.

The target column is `label`.


## Checking the target

Since this is a classification problem, I want to check how many benign and phishing URLs are in the dataset.


In [ ]:
label_counts = df["label"].value_counts().sort_index()
label_counts


In [ ]:
label_percent = (
    df["label"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

label_percent.round(2)


There is a noticeable class imbalance in the dataset.

About 85.8% of the URLs are benign and about 14.2% are phishing. Because of this, accuracy by itself may not be enough to judge our models later. Recall, precision, F1 score, and ROC AUC will also be important.


In [ ]:
plt.figure(figsize=(6, 4))

label_counts.plot(kind="bar")

plt.title("Benign vs Phishing URLs")
plt.xlabel("Label")
plt.ylabel("Number of URLs")
plt.xticks(
    [0, 1],
    ["Benign (0)", "Phishing (1)"],
    rotation=0
)

plt.show()


The graph makes the class imbalance pretty clear. There are many more benign URLs than phishing URLs, so this is something we will need to keep in mind during model evaluation.


## Checking missing values

Before using the data, I want to see whether any columns have missing values.


In [ ]:
missing_values = df.isnull().sum()

missing_values[
    missing_values > 0
]


The dataset only has a very small number of missing values. These occur in the `tld` text column, so they will not affect the first numeric version of the dataset.


## Checking duplicate rows

I also want to check whether the same observations appear more than once.


In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)


There are duplicate rows in the dataset. Since duplicated observations could affect model training and evaluation, I will remove them before building the first modeling dataset.


In [ ]:
df_clean = df.drop_duplicates().copy()

print("Original rows:", len(df))
print("Rows after removing duplicates:", len(df_clean))


## Basic numeric summary

Now I want to look at the range and average values of the numerical features to see whether any of them are on very different scales.


In [ ]:
df_clean.describe().T


The numerical features use different ranges. Ratio features are mostly between 0 and 1 while length and count features can be much larger.

Because some machine learning algorithms are sensitive to feature scale, scaling the numerical predictors will be useful later.


## Comparing a few features by class

I want to see whether some basic URL characteristics look different between benign and phishing URLs.


In [ ]:
features_to_compare = [
    "url_len",
    "dom_len",
    "digit_cnt",
    "special_cnt",
    "entropy",
    "path_len"
]

df_clean.groupby("label")[features_to_compare].mean().round(3)


The averages give an early look at whether phishing URLs differ from benign URLs in characteristics such as URL length, number of digits, special characters, and entropy.

These differences suggest that the engineered URL features may provide useful information for classification.


## Correlation with the target

Since most of the dataset is already numerical, I can also check which numerical features have the strongest relationship with the label.


In [ ]:
numeric_df = df_clean.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
)

target_corr = (
    numeric_df
    .corr()["label"]
    .drop("label")
    .sort_values(
        key=abs,
        ascending=False
    )
)

target_corr


In [ ]:
plt.figure(figsize=(8, 6))

target_corr.head(10).sort_values().plot(
    kind="barh"
)

plt.title("Features Most Correlated With the Label")
plt.xlabel("Correlation")

plt.show()


Correlation does not automatically mean that a feature will be important to every model, but this gives us an early idea of which numerical features may contain useful information for phishing detection.


## Making a basic numeric dataset

The raw `url`, `dom`, and `tld` columns contain text and have many unique values. Since this dataset already provides engineered numerical features from those strings, I am leaving the raw text columns out of the first numeric version.

We can come back later and engineer or encode those fields if they improve model performance.


In [ ]:
text_columns = [
    "url",
    "dom",
    "tld"
]

model_df = df_clean.drop(
    columns=text_columns
)

model_df.head()


In [ ]:
print("Modeling rows:", model_df.shape[0])
print("Modeling columns:", model_df.shape[1])

print(
    "Missing numeric values:",
    model_df.isnull().sum().sum()
)


After removing the three raw text columns, the first modeling version contains 22 numerical predictors and the label.

There should be no missing values in this numeric version.


## Separating predictors and target

Before scaling, I am separating the features from the target because the label should not be scaled.


In [ ]:
X = model_df.drop(
    columns=["label"]
)

y = model_df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## Scaling the numerical features

For this first EDA version, I am standardizing the predictor variables so that features with much larger values do not dominate algorithms that are sensitive to scale.

The actual reusable preprocessing steps will be moved into our `dataset_preparation.py` pipeline later.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

X_scaled.head()


In [ ]:
X_scaled.describe().T[
    ["mean", "std", "min", "max"]
].round(3)


After standardization, the predictors are now on a more comparable scale. This will be especially useful for models such as Logistic Regression and SVM.

Tree-based models such as Decision Trees and Random Forests do not depend on scaling in the same way, so our later preprocessing pipeline can handle models differently if needed.


## EDA Summary

The dataset contains 116,600 URL observations and 26 original columns.

The target is binary:
- 0 = benign
- 1 = phishing

The dataset is imbalanced, with many more benign URLs than phishing URLs. There are also duplicate rows and a very small number of missing values in the `tld` text column.

Most of the useful features are already numerical URL characteristics. For the first modeling version, I removed the raw `url`, `dom`, and `tld` fields and kept the 22 engineered numerical predictors.

The numerical predictors were also standardized so that the data is ready for an initial round of machine learning models. More preprocessing and feature changes can be added later if the model results show that they are needed.

**Important:** In the final preprocessing pipeline, the train/dev/test split should happen before fitting the scaler so that there is no data leakage.
